# WaveGrip: Test-set evaluation of the gesture-classification CNNs

Tristan Au

This notebook evaluates three trained ResNet-152 v1.5 models in `CNN/Runs/` (Raw sEMG, PSD and DWT time-frequency inputs) on the **held-out test set**. The training notebooks (`CNN/cnn_raw.ipynb`, `CNN/cnn_psd.ipynb`, `CNN/cnn_dwt_tf_image.ipynb`) never touch it. All three models share the same 3,071 test trials (20 % of the 15,351 GRABMyo wrist trials, stratified by participant × gesture).

**What it shows**

1. A confusion matrix for each CNN. Cells are coloured by row-normalised % (each true gesture's row sums to 100 %) and annotated with the trial count and the %.
2. A table comparing the CNNs on accuracy, plus macro-averaged precision, recall and F1-score (the unweighted mean over the 17 gestures).
3. ROC curves, one-vs-rest with macro-averaging: one figure overlaying the three CNNs' macro-average curves, and one panel per CNN showing its 17 per-class curves under the macro-average.

**This notebook only reads `CNN/Runs/` and `ProcessedDatasets/`.** It never writes there. Outputs go to `Evaluation/outputs/`.

### Rebuilding missing run files

The three run folders are not equally complete. Each missing piece is rebuilt exactly from the training code's own definitions:

| Run | Missing | How it is rebuilt here |
|---|---|---|
| `raw_random_resnet152` | `*_split.npz` | Same two-step stratified split as training (`train_test_split`, seed 0 then 1). It is checked to equal the saved DWT/PSD splits. |
| `raw_random_resnet152` | `*_normalization.npz` | Per-channel mean/std in mV over the **training rows only**, same as `RawInputs`. |
| `psd_random_resnet152` | `*_state_dict.pt` | `best_state` from `*_checkpoint.pt`. That is what training saves as `*_state_dict.pt`; for the raw and DWT runs the two are identical. |
| `psd_random_resnet152` | `*_normalization.npz` | Per-(channel, bin) mean/std of log10(PSD) over the **training rows only**, same as `load_psd`. |
| `dwttf_random_resnet152` | nothing | The saved files are used as they are. |

Whether a model uses skip connections is read from its weights: the `downsample` projection layers only exist in the skip variant. The number of input channels (1 for raw and PSD, 12 for the time-frequency image) is read from each run's `conv1` weights.

The DWT model here is the **time-frequency** CNN (`CNN/cnn_dwt_tf_image.ipynb`, input 12 × 5 × 2560: electrodes as channels × bands A5–D2 × a continuous 1.95 ms time grid). The earlier joined-band DWT CNN (1 × 12 × 5120) is compared against it in `dwt_cnns_evaluation.ipynb`.

**Precision note.** Training ran on a T4 with float16 autocast. Here inference runs in float32 (on CPU or GPU, whichever is available), so the predicted probabilities may differ from training-time outputs in the last few decimals. The DWT time-frequency inputs are rounded through float16 because training kept them on the GPU in that format.

### Running on Google Colab (T4 GPU)

1. Google Drive needs the same layout the training notebooks use under `MyDrive/WaveGrip/`:
   - `ProcessedDatasets/raw_wrist/grabmyo_wrist_raw.h5`, `ProcessedDatasets/psd_wrist/grabmyo_wrist_psd.h5`, `ProcessedDatasets/dwt_wrist/grabmyo_wrist_dwt.h5`
   - `CNN/Runs/` (or `CNN/runs/`, where the training notebooks save) with the three run folders `raw_random_resnet152/`, `psd_random_resnet152/`, `dwttf_random_resnet152/`. A run folder may also sit one level deeper, e.g. inside a folder created by a Drive download (`dwttf_random_resnet152-2026…/dwttf_random_resnet152/`); the notebook finds it either way.
2. Open this notebook in Colab: **Runtime → Change runtime type → T4 GPU**.
3. Run all cells. The configuration cell mounts Drive and copies the three datasets (~7.8 GB) to the Colab machine's local disk once per session. Outputs go to `MyDrive/WaveGrip/Evaluation/outputs/`, so they outlive the session.

The notebook also runs locally from `Evaluation/` with no changes, but CPU inference of the raw model takes well over an hour.

### Caching

Test-set probabilities are saved to `outputs/<run>_test_probs.npz` after inference. Re-running the notebook loads them and skips inference. The DWT time-frequency cache is the same file `dwt_cnns_evaluation.ipynb` writes, so whichever notebook runs first saves the other one the inference. Set `FORCE_INFERENCE = True` to recompute.

## 0. Configuration

In [ ]:
import gc
import shutil
import sys
import time
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LinearSegmentedColormap
from sklearn.metrics import (accuracy_score, auc, confusion_matrix, f1_score, precision_score,
                             recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from torch import nn

FORCE_INFERENCE = False                              # True: ignore cached test probabilities
DATA_FILES = ["raw_wrist/grabmyo_wrist_raw.h5", "psd_wrist/grabmyo_wrist_psd.h5",
              "dwt_wrist/grabmyo_wrist_dwt.h5"]      # relative to ProcessedDatasets/

# ---- Paths: Google Colab (Drive) or the local project folder --------------------------------------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/WaveGrip")    # same layout as the local project folder
    EVAL_DIR = PROJECT_ROOT / "Evaluation"
    DATA_DIR = Path("/content/data")                           # Drive reads are slow: copy once per session
    for rel in DATA_FILES:
        src, dst = PROJECT_ROOT / "ProcessedDatasets" / rel, DATA_DIR / rel
        assert src.exists(), f"upload the dataset to {src}"
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            dst.parent.mkdir(parents=True, exist_ok=True)
            t0 = time.perf_counter()
            shutil.copyfile(src, dst)
            print(f"copied {src.name} to local disk in {time.perf_counter() - t0:.0f} s")
else:
    PROJECT_ROOT = Path.cwd().parent                           # run from Evaluation/
    EVAL_DIR = Path.cwd()
    DATA_DIR = PROJECT_ROOT / "ProcessedDatasets"              # read only
RUN_ROOTS = [PROJECT_ROOT / "CNN" / d for d in ["Runs", "runs"] if (PROJECT_ROOT / "CNN" / d).is_dir()]
assert RUN_ROOTS, f"no CNN/Runs (or CNN/runs) folder under {PROJECT_ROOT}"   # read only
for rel in DATA_FILES:
    assert (DATA_DIR / rel).exists(), f"dataset not found at {DATA_DIR / rel}"
OUT_DIR = EVAL_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Device ---------------------------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True                      # fastest conv algorithms for fixed shapes
elif IN_COLAB:
    print("WARNING: no GPU. In Colab use Runtime -> Change runtime type -> T4 GPU")

# ---- The three CNNs (fixed order and fixed colour everywhere) -------------------------------------
MODELS = {
    "raw": dict(label="Raw sEMG", run="raw_random_resnet152", data="raw_wrist/grabmyo_wrist_raw.h5",
                input_shape=(1, 12, 10240), batch=16, color="#2a78d6"),
    "psd": dict(label="PSD", run="psd_random_resnet152", data="psd_wrist/grabmyo_wrist_psd.h5",
                input_shape=(1, 12, 246), batch=64, color="#eb6834"),
    "dwttf": dict(label="DWT time-frequency", run="dwttf_random_resnet152", data="dwt_wrist/grabmyo_wrist_dwt.h5",
                  input_shape=(12, 5, 2560), batch=32, color="#1baf7a"),
}


def find_run(run):
    # CNN/Runs/<run>/ or CNN/runs/<run>/, or one level deeper (e.g. a Drive-download folder <run>-2026.../<run>/)
    for base in RUN_ROOTS:
        hits = [base / run] if (base / run).is_dir() else sorted(p.parent for p in base.glob(f"*/{run}/{run}_*.pt"))
        if hits:
            return hits[0]
    raise FileNotFoundError(f"run folder {run}/ not found under {[str(b) for b in RUN_ROOTS]}")


for m in MODELS.values():
    m["dir"] = find_run(m["run"])

# ---- Training settings the evaluation must match (from the training notebooks) --------------------
N_CLASSES = 17
SPLIT, TEST_FRACTION, VAL_FRACTION, SEED = "random", 0.2, 0.2, 0
RESNET_LAYERS = (3, 8, 36, 3)
RESNET_WIDTHS = (64, 128, 256, 512)
DWT_BANDS = ["A5", "D5", "D4", "D3", "D2"]           # time-frequency image rows (low -> high frequency)
N_TIME = 2560                                        # time-frequency grid: 1.95 ms per column
CHUNK = 512                                          # rows per block when streaming the HDF5 files

# GRABMyo gesture names (Dataset/.../MotionSequence.txt), gestures 1-17 -> classes 0-16
GESTURE_NAMES = ["Lateral Prehension", "Thumb Adduction", "Thumb and Little Finger Opposition",
                 "Thumb and Index Finger Opposition", "Thumb and Index Finger Extension",
                 "Thumb and Little Finger Extension", "Index and Middle Finger Extension",
                 "Little Finger Extension", "Index Finger Extension", "Thumb Finger Extension",
                 "Wrist Extension", "Wrist Flexion", "Forearm Supination", "Forearm Pronation",
                 "Hand Open", "Hand Close", "Rest"]
CLASS_TICKS = [str(i) for i in range(1, N_CLASSES + 1)]

# ---- Plot style (palette of the training notebooks' curve plots) ----------------------------------
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, PER_CLASS_GRAY = "#e5e4e0", "#c3c2b7", "#c3c2b7"
CM_CMAP = LinearSegmentedColormap.from_list(                 # one-hue blue ramp, 0 % -> 100 %
    "blue_seq", ["#fcfcfb", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])
plt.rcParams.update({"font.family": "sans-serif", "axes.edgecolor": AXIS, "axes.labelcolor": INK_2,
                     "xtick.color": INK_2, "ytick.color": INK_2, "axes.titlecolor": INK})

for key, m in MODELS.items():
    print(f"{key:>5}: {m['dir'].relative_to(PROJECT_ROOT)}  files: {sorted(p.name.removeprefix(m['run'] + '_') for p in m['dir'].iterdir())}")
gpu = f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""
print(f"runs: {[str(b) for b in RUN_ROOTS]}\ndata: {DATA_DIR}\ndevice {device}{gpu}, torch {torch.__version__}, outputs -> {OUT_DIR}")

## 1. Test set

The index arrays (`session`, `participant`, `gesture`, `trial`) are identical in all three dataset files, so a row index means the same trial everywhere. The DWT time-frequency and PSD runs saved their split. The raw run did not, so the split is rebuilt with the training code's `holdout` and checked against the saved ones. All three must share one test set.

In [ ]:
def read_meta(path):
    with h5py.File(path) as f:
        return {k: f[k][:].astype(np.int64) for k in ["session", "participant", "gesture", "trial"]}


meta = read_meta(DATA_DIR / MODELS["raw"]["data"])
for key in ["psd", "dwttf"]:
    other = read_meta(DATA_DIR / MODELS[key]["data"])
    assert all(np.array_equal(meta[k], other[k]) for k in meta), f"{key} dataset rows differ from raw"
labels = meta["gesture"] - 1                                              # classes 0-16


def holdout(rows, fraction, seed):
    # Copy of the training notebooks' "random" split: stratified by participant x gesture
    strata = meta["participant"][rows] * 100 + meta["gesture"][rows]
    return train_test_split(rows, test_size=fraction, stratify=strata, random_state=seed)


trainval_rows, test_rows = holdout(np.arange(len(labels)), TEST_FRACTION, SEED)
train_rows, val_rows = holdout(trainval_rows, VAL_FRACTION, SEED + 1)
rebuilt = {"train": np.sort(train_rows), "val": np.sort(val_rows), "test": np.sort(test_rows)}

splits = {}
for key, m in MODELS.items():
    path = m["dir"] / f"{m['run']}_split.npz"
    if path.exists():
        saved = np.load(path)
        assert all(np.array_equal(saved[k], rebuilt[k]) for k in rebuilt), f"{key}: saved split != rebuilt split"
        splits[key] = "saved (= rebuilt)"
    else:
        splits[key] = "rebuilt"

TRAIN_ROWS, TEST_ROWS = rebuilt["train"], rebuilt["test"]
y_test = labels[TEST_ROWS]
print("split source:", splits)
print(f"train {len(TRAIN_ROWS)}, val {len(rebuilt['val'])}, test {len(TEST_ROWS)} trials; "
      f"test trials per gesture {np.bincount(y_test).min()}-{np.bincount(y_test).max()}")

## 2. Model: ResNet-152 v1.5

`Bottleneck` and `ResNet152` are copied from the training notebooks (Section 3) so the saved `state_dict` keys line up. The skip-connection setting is a constructor argument here and is detected from each run's weights; the number of input channels is read from `conv1`.

In [ ]:
class Bottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_ch, width, stride=1, skip=True):
        super().__init__()
        out_ch = width * self.expansion
        self.conv1 = nn.Conv2d(in_ch, width, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(width)
        self.conv2 = nn.Conv2d(width, width, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(width)
        self.conv3 = nn.Conv2d(width, out_ch, 1, bias=False)
        self.bn3 = nn.BatchNorm2d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.skip = skip
        self.downsample = None
        if skip and (stride != 1 or in_ch != out_ch):
            self.downsample = nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                                            nn.BatchNorm2d(out_ch))

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        if self.skip:
            out = out + (x if self.downsample is None else self.downsample(x))
        return self.relu(out)


class ResNet152(nn.Module):
    def __init__(self, in_ch=1, n_classes=N_CLASSES, layers=RESNET_LAYERS, widths=RESNET_WIDTHS, skip=True):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 64, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        ch, stages = 64, []
        for i, (n_blocks, width) in enumerate(zip(layers, widths)):
            blocks = []
            for b in range(n_blocks):
                blocks.append(Bottleneck(ch, width, stride=2 if (i > 0 and b == 0) else 1, skip=skip))
                ch = width * Bottleneck.expansion
            stages.append(nn.Sequential(*blocks))
        self.layer1, self.layer2, self.layer3, self.layer4 = stages
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(ch, n_classes)
        self.log_softmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.log_softmax(self.fc(torch.flatten(self.avgpool(x), 1)))


def load_trained(key):
    # Weights from *_state_dict.pt, or from best_state in *_checkpoint.pt if the state_dict is missing
    m = MODELS[key]
    sd_path, ckpt_path = m["dir"] / f"{m['run']}_state_dict.pt", m["dir"] / f"{m['run']}_checkpoint.pt"
    if sd_path.exists():
        state, source = torch.load(sd_path, map_location="cpu", weights_only=True), sd_path.name
    else:
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=True, mmap=True)
        assert ckpt["finished"], f"{key}: checkpoint is from an unfinished run"
        state, source = ckpt["best_state"], f"{ckpt_path.name} [best_state, epoch {ckpt['best_epoch']}]"
        del ckpt
    in_ch = state["conv1.weight"].shape[1]
    assert in_ch == m["input_shape"][0], f"{key}: conv1 has {in_ch} input channels, expected {m['input_shape'][0]}"
    skip = any(".downsample." in k for k in state)
    model = ResNet152(in_ch=in_ch, skip=skip)
    model.load_state_dict(state)
    m["weights_source"], m["skip"] = source, skip
    return model.to(device).eval()

## 3. Test inputs

Each loader returns a function `get(i)` that gives the normalised test inputs for test positions `i` as float32: `(B, 1, 12, W)` for raw and PSD, `(B, 12, 5, 2560)` for the DWT time-frequency image. The normalisation statistics always come from the **training rows** (saved, or recomputed exactly as in training), never from the test set.

| Input | Preprocessing (same as training) |
|---|---|
| Raw | counts → mV `(counts − baseline) / gain` → z-score per channel |
| PSD | `log10(psd + 1e-12)` → z-score per (channel, bin) |
| DWT time-frequency | z-score each band per electrode with the saved statistics → float16 round trip → repeat each band onto the shared 2560-column grid (A5/D5 × 8, D4 × 4, D3 × 2, D2 × 1) → `(B, 12, 5, 2560)` |

In [ ]:
def iter_blocks(n_rows):
    for start in range(0, n_rows, CHUNK):
        yield start, min(start + CHUNK, n_rows)


def rows_mask(rows, n_rows):
    mask = np.zeros(n_rows, bool)
    mask[rows] = True
    return mask


def load_raw_inputs(m):
    # One pass over the file: training-row statistics (float64, in mV) + the test rows' int16 counts
    is_train, is_test = rows_mask(TRAIN_ROWS, len(labels)), rows_mask(TEST_ROWS, len(labels))
    with h5py.File(DATA_DIR / m["data"]) as f:
        gain, baseline = f["gain"][:], f["baseline"][:].astype(np.float64)
        s = ss = 0.0
        n = 0
        test_counts = []
        for a, b in iter_blocks(len(labels)):
            block = f["emg"][a:b]                                          # (rows, 12, 10240) int16
            tr = is_train[a:b]
            if tr.any():
                x = (block[tr] - baseline[a:b][tr][..., None]) / gain[a:b][tr][..., None]
                s, ss, n = s + x.sum(axis=(0, 2)), ss + (x ** 2).sum(axis=(0, 2)), n + x.shape[0] * x.shape[2]
            test_counts.append(block[is_test[a:b]])
    mean = s / n
    std = np.sqrt(np.maximum(ss / n - mean ** 2, 0))
    std = np.where(std > 0, std, 1.0)
    counts = torch.from_numpy(np.concatenate(test_counts))                 # test rows, in TEST_ROWS order
    g, bl = gain[TEST_ROWS], baseline[TEST_ROWS]
    # ((counts - baseline) / gain - mean) / std  ==  counts * a + b   (as in training's RawInputs)
    sc = torch.from_numpy((1.0 / (g * std)).astype(np.float32))
    of = torch.from_numpy((-(bl / g + mean) / std).astype(np.float32))
    return lambda i: (counts[i].float() * sc[i][..., None] + of[i][..., None]).unsqueeze(1)


def load_psd_inputs(m):
    with h5py.File(DATA_DIR / m["data"]) as f:
        x = np.log10(f["psd"][:].astype(np.float64) + 1e-12)              # (N, 12, 246)
    mean, std = x[TRAIN_ROWS].mean(axis=0), x[TRAIN_ROWS].std(axis=0)      # per (channel, bin)
    std = np.where(std > 0, std, 1.0)
    xt = torch.from_numpy(((x[TEST_ROWS] - mean) / std).astype(np.float32))
    return lambda i: xt[i].unsqueeze(1)


def tf_index(lengths, n_time=N_TIME):
    """Copy of cnn_dwt_tf_image.ipynb: (n_bands, n_time) column of the joined 5120 layout for each image pixel."""
    rows, off = [], 0
    for L in lengths:
        assert n_time % L == 0, (L, n_time)
        rows.append(off + np.arange(n_time) * L // n_time)                 # repeat each coefficient n_time / L times
        off += L
    return np.stack(rows)


def load_dwttf_inputs(m):
    norm = np.load(m["dir"] / f"{m['run']}_normalization.npz")
    is_test = rows_mask(TEST_ROWS, len(labels))
    parts = []
    with h5py.File(DATA_DIR / m["data"]) as f:
        for band in DWT_BANDS:
            x = np.concatenate([f["dwt"][band][a:b][is_test[a:b]] for a, b in iter_blocks(len(labels))])
            x = x.astype(np.float32)
            x -= norm[f"{band}_mean"][:, None].astype(np.float32)
            x /= norm[f"{band}_std"][:, None].astype(np.float32)
            parts.append(x)
    idx = torch.from_numpy(tf_index([p.shape[-1] for p in parts]))        # (5, 2560)
    xt = torch.from_numpy(np.concatenate(parts, axis=-1)).to(torch.float16)   # (n_test, 12, 5120); training kept it as float16
    return lambda i: xt[i][:, :, idx].float()                              # (B, 12, 5, 2560)


INPUT_LOADERS = {"raw": load_raw_inputs, "psd": load_psd_inputs, "dwttf": load_dwttf_inputs}

## 4. Inference on the test set

For each CNN this loads the test probabilities from the cache if they exist; otherwise it builds the test inputs, loads the weights and runs the model. The model's output is log-softmax, so the class probabilities are `exp(output)`.

In [ ]:
@torch.inference_mode()
def predict(model, get, n, batch):
    out, t0 = [], time.perf_counter()
    for k, start in enumerate(range(0, n, batch)):
        i = torch.arange(start, min(start + batch, n))
        out.append(model(get(i).to(device)).float().exp().cpu())
        if (k + 1) % 20 == 0 or start + batch >= n:
            done = min(start + batch, n)
            print(f"\r  {done}/{n} trials, {time.perf_counter() - t0:.0f} s", end="", flush=True)
    print()
    return torch.cat(out).numpy()


probs = {}
for key, m in MODELS.items():
    cache = OUT_DIR / f"{m['run']}_test_probs.npz"
    if cache.exists() and not FORCE_INFERENCE:
        c = np.load(cache, allow_pickle=False)
        assert np.array_equal(c["test_rows"], TEST_ROWS), f"{cache.name} was made for a different test set"
        probs[key] = c["probs"]
        m["weights_source"], m["skip"] = str(c["weights_source"]), bool(c["skip"])
        print(f"{key}: loaded cached probabilities from {cache.name}")
        continue
    print(f"{key}: building test inputs ...")
    t0 = time.perf_counter()
    get = INPUT_LOADERS[key](m)
    x0 = get(torch.arange(2))
    assert tuple(x0.shape[1:]) == m["input_shape"] and torch.isfinite(x0).all(), x0.shape
    print(f"  inputs ready in {time.perf_counter() - t0:.0f} s")
    model = load_trained(key)
    print(f"  weights: {m['weights_source']} (skip connections: {m['skip']})")
    probs[key] = predict(model, get, len(TEST_ROWS), m["batch"])
    np.savez(cache, probs=probs[key], test_rows=TEST_ROWS, y_true=y_test,
             weights_source=m["weights_source"], skip=m["skip"])
    del model, get
    gc.collect()

for key, p in probs.items():
    assert p.shape == (len(TEST_ROWS), N_CLASSES) and np.allclose(p.sum(1), 1, atol=1e-3), key
preds = {key: p.argmax(1) for key, p in probs.items()}

## 5. Gesture key

The plots label the gestures 1–17. This is the GRABMyo name for each number.

In [ ]:
gesture_key = pd.DataFrame({"Gesture": range(1, N_CLASSES + 1), "Name": GESTURE_NAMES,
                            "Test trials": np.bincount(y_test, minlength=N_CLASSES)}).set_index("Gesture")
gesture_key

## 6. Confusion matrices

One figure per CNN. Rows are the true gesture and columns the predicted gesture. Colour shows the row-normalised % (each row sums to 100 %, so the diagonal is per-gesture recall). Each cell shows the trial count over its %; cells with no trials show a muted `0`.

In [ ]:
def plot_confusion(key):
    m = MODELS[key]
    cm = confusion_matrix(y_test, preds[key], labels=range(N_CLASSES))
    pct = 100 * cm / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(13, 11.5), constrained_layout=True)
    im = ax.imshow(pct, cmap=CM_CMAP, vmin=0, vmax=100)
    for r in range(N_CLASSES):
        for c in range(N_CLASSES):
            if cm[r, c] == 0:
                ax.text(c, r, "0", ha="center", va="center", fontsize=7.5, color=MUTED)
            else:
                ax.text(c, r, f"{cm[r, c]}\n{pct[r, c]:.1f}%", ha="center", va="center", fontsize=7.5,
                        linespacing=1.15, color="#ffffff" if pct[r, c] >= 55 else INK)
    ax.set_xticks(range(N_CLASSES), CLASS_TICKS)
    ax.set_yticks(range(N_CLASSES), CLASS_TICKS)
    ax.set_xlabel("Predicted gesture")
    ax.set_ylabel("True gesture")
    ax.set_xticks(np.arange(-0.5, N_CLASSES), minor=True)
    ax.set_yticks(np.arange(-0.5, N_CLASSES), minor=True)
    ax.grid(which="minor", color="#ffffff", lw=1)                         # thin gap between cells
    ax.tick_params(which="minor", length=0)
    ax.spines[:].set_visible(False)
    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label("% of true-gesture trials")
    cbar.outline.set_visible(False)
    acc = (preds[key] == y_test).mean()
    ax.set_title(f"{m['label']} ResNet-152 ({' × '.join(map(str, m['input_shape']))} input): confusion matrix on the test set "
                 f"(n = {len(y_test)}, accuracy {acc:.3f})", loc="left", fontsize=13)
    fig.savefig(OUT_DIR / f"{m['run']}_confusion_matrix.png", dpi=150)
    plt.show()

In [ ]:
plot_confusion("raw")

In [ ]:
plot_confusion("psd")

In [ ]:
plot_confusion("dwttf")

## 7. Metrics table

Accuracy, plus macro-averaged precision, recall and F1-score (the unweighted mean of the 17 per-gesture scores), on the test set. The best value in each column is in bold. The table is also saved as `outputs/cnn_test_metrics.csv`.

In [ ]:
metrics = pd.DataFrame({
    MODELS[key]["label"]: {
        "Accuracy": accuracy_score(y_test, preds[key]),
        "Precision (macro)": precision_score(y_test, preds[key], average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, preds[key], average="macro", zero_division=0),
        "F1-score (macro)": f1_score(y_test, preds[key], average="macro", zero_division=0),
    } for key in MODELS}).T
metrics.index.name = "CNN"
metrics.to_csv(OUT_DIR / "cnn_test_metrics.csv", float_format="%.6f")
metrics.style.format("{:.4f}").highlight_max(axis=0, props="font-weight: bold")

## 8. ROC curves: one-vs-rest, macro-averaged

For each gesture, the CNN's probability for that gesture is scored as a binary classifier (that gesture vs. the other 16). This gives 17 one-vs-rest ROC curves. The **macro-average ROC curve** is the mean of the 17 TPRs, each linearly interpolated onto the union of their FPR points. The **macro AUC** in the legends is `roc_auc_score(..., multi_class="ovr", average="macro")`, the unweighted mean of the 17 per-class AUCs. The area under the averaged curve is printed alongside it; the two agree closely but are not defined identically.

In [ ]:
Y_test = label_binarize(y_test, classes=range(N_CLASSES))
roc = {}
for key, p in probs.items():
    per_class = [roc_curve(Y_test[:, c], p[:, c])[:2] for c in range(N_CLASSES)]
    fpr_grid = np.unique(np.concatenate([fpr for fpr, _ in per_class]))
    mean_tpr = np.mean([np.interp(fpr_grid, fpr, tpr) for fpr, tpr in per_class], axis=0)
    roc[key] = dict(per_class=per_class, fpr=fpr_grid, tpr=mean_tpr,
                    auc=roc_auc_score(y_test, p, multi_class="ovr", average="macro"),
                    curve_auc=auc(fpr_grid, mean_tpr))
    print(f"{MODELS[key]['label']:<19} macro AUC (mean of per-class AUCs) {roc[key]['auc']:.4f} | "
          f"area under macro-average curve {roc[key]['curve_auc']:.4f}")


def style_roc_axes(ax):
    ax.plot([0, 1], [0, 1], color=MUTED, ls="--", lw=1, label="Chance (AUC = 0.5)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.01)
    ax.set_aspect("equal")
    ax.set_xlabel("False positive rate")
    ax.set_ylabel("True positive rate")
    ax.grid(color=GRID, lw=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)

### 8a. The three CNNs compared

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6.5), constrained_layout=True)
for key, r in roc.items():
    ax.plot(r["fpr"], r["tpr"], color=MODELS[key]["color"], lw=2,
            label=f"{MODELS[key]['label']} (macro AUC = {r['auc']:.4f})")
style_roc_axes(ax)
ax.legend(frameon=False, loc="lower right")
ax.set_title("Macro-averaged one-vs-rest ROC on the test set", loc="left")
fig.savefig(OUT_DIR / "cnn_roc_macro_ovr_comparison.png", dpi=150)
plt.show()

### 8b. Per-class curves for each CNN

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6.5), constrained_layout=True)
for ax, (key, r) in zip(axes, roc.items()):
    for c, (fpr, tpr) in enumerate(r["per_class"]):
        ax.plot(fpr, tpr, color=PER_CLASS_GRAY, lw=1, label="Per-class OvR (gestures 1-17)" if c == 0 else None)
    ax.plot(r["fpr"], r["tpr"], color=MODELS[key]["color"], lw=2.5, label=f"Macro-average (AUC = {r['auc']:.4f})")
    style_roc_axes(ax)
    ax.legend(frameon=False, loc="lower right")
    ax.set_title(MODELS[key]["label"], loc="left")
fig.suptitle("One-vs-rest ROC per gesture and macro-average, test set", x=0.01, ha="left", fontsize=13)
fig.savefig(OUT_DIR / "cnn_roc_ovr_per_class.png", dpi=150)
plt.show()

## 9. Saved outputs

In [ ]:
for key, m in MODELS.items():
    print(f"{m['label']:<19} weights: {m['weights_source']} (skip connections: {m['skip']})")
print(f"\nin {OUT_DIR}:")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:<45} {p.stat().st_size / 1e3:9.1f} kB")